# AE recurrent league-MAPPO training notebook

This notebook wraps `ae/training/train_recurrent_mappo.py`. It trains the recurrent masked actor used by `ae/src/ae_manager.py` and exports the checkpoint to `ae/src/artifacts/ae_actor.pt`.

Run from the repository root or from this notebook's folder. Initialize submodules first with `git submodule update --init`.


In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if CWD.name == 'training':
    REPO = CWD.parents[1]
elif CWD.name == 'ae':
    REPO = CWD.parent
else:
    REPO = CWD

sys.path.insert(0, str(REPO / 'ae' / 'training'))
sys.path.insert(0, str(REPO / 'ae' / 'src'))
sys.path.insert(0, str(REPO / 'til-26-ae'))

print('Repo:', REPO)


## Install dependencies if needed

Uncomment the cell below in a fresh Workbench environment if `til_environment`, `torch`, or `tqdm` are missing.


In [ ]:
# %pip install -e {REPO / 'til-26-ae'} torch numpy matplotlib tqdm


In [ ]:
from argparse import Namespace
import train_recurrent_mappo as trainer

args = Namespace(
    updates=300,
    episodes_per_update=16,
    eval_every=5,
    eval_episodes=6,
    seed=26,
    cpu=False,
    torch_threads=1,
    # Opponent mixture: keep random/noisy small; use it as robustness data only.
    p_selfplay=0.35,
    p_heuristic=0.55,
    p_random=0.10,
)
args


In [ ]:
trainer.train(args)


## After training

The best checkpoint is saved to `ae/src/artifacts/ae_actor.pt`. The AE Dockerfile copies `ae/src`, so the checkpoint will be included in the image. Build and test after the checkpoint exists:

```bash
cd ae
docker build -t TEAM_ID-ae:finals .
```

Next high-impact upgrades: add your previous best MAPPO checkpoint as a frozen opponent in `opponent_action`, and replace `ValueNet` with a centralized critic during training only.
